# COMP842 Exercise 2 - Proof of Work: Mining, Difficulty and Probability

In [8]:
# Import SHA-256, timing, and table-processing libraries.
import hashlib
import time
import pandas as pd

def mine_block(block_data, difficulty):
    # A valid hash must begin with the required number of zeros.
    target_prefix = "0" * difficulty
    nonce = 0
    attempts = 0
    start = time.perf_counter()

    # Try different nonces until a valid hash is found.
    while True:
        candidate = f"{block_data}|{nonce}"
        block_hash = hashlib.sha256(candidate.encode("utf-8")).hexdigest()
        attempts += 1

        if block_hash.startswith(target_prefix):
            elapsed = time.perf_counter() - start
            return nonce, block_hash, attempts, elapsed

        nonce += 1

# Test four difficulty levels with 30 blocks each.
difficulties = [2, 3, 4, 5]
runs_per_difficulty = 30
records = []

for difficulty in difficulties:
    print(f"\n{'='*25} DIFFICULTY {difficulty} {'='*25}")
    for run in range(1, runs_per_difficulty + 1):
        # Use unique data so each mining run has a new search problem.
        block_data = (
            f"COMP842|difficulty={difficulty}|run={run}|"
            f"unique={time.time_ns()}"
        )

        nonce, valid_hash, attempts, elapsed = mine_block(
            block_data, difficulty
        )

        # Store the measurements needed for the performance analysis.
        records.append({
            "Difficulty": difficulty,
            "Run": run,
            "Nonce": nonce,
            "Hash Attempts": attempts,
            "Mining Time (s)": elapsed,
            "Valid Hash": valid_hash,
        })

        print(
            f"Run {run:02d} | nonce={nonce:<10} | "
            f"attempts={attempts:<10} | time={elapsed:.6f}s | "
            f"hash={valid_hash}"
        )

# Convert all 120 mining runs into a DataFrame.
runs_df = pd.DataFrame(records)



========================= DIFFICULTY 2 =========================
Run 01 | nonce=174        | attempts=175        | time=0.000238s | hash=0017817d388aaa1bb8264f689ac0f2a61eb0e534ad92229fbe65a9d1e11f09b8
Run 02 | nonce=700        | attempts=701        | time=0.000929s | hash=0031d99d004c4473f9285e98588871aad81318de3aa33006ac2073ddce30ebe6
Run 03 | nonce=21         | attempts=22         | time=0.000041s | hash=00a250d7bb2ae9aeddb55e9654e299e43d7efaa1976a1eeb384b88f6a4f4e64a
Run 04 | nonce=293        | attempts=294        | time=0.000382s | hash=00255334e19dc222515545f167291b2a7cb697ef078ee80f76538abb6c4f2374
Run 05 | nonce=27         | attempts=28         | time=0.000029s | hash=00eefb955715f67feafec7bb333c85ac57ab4a9c73821a23e926f5343431f589
Run 06 | nonce=384        | attempts=385        | time=0.000395s | hash=0089be2a09cb8d66c7b8ae9900c3ebab4388397e1e2193b5bd4690dbc23a9f10
Run 07 | nonce=217        | attempts=218        | time=0.000215s | hash=00bbd3d8dc900318fa6ebc6e9429b21f9769746f

In [9]:
# Calculate the required performance metrics for each difficulty.
summary_rows = []

for difficulty in difficulties:
    subset = runs_df[runs_df["Difficulty"] == difficulty]

    measured_average_attempts = subset["Hash Attempts"].mean()
    theoretical_attempts = 16 ** difficulty
    total_attempts = subset["Hash Attempts"].sum()
    total_time = subset["Mining Time (s)"].sum()

    summary_rows.append({
        "Difficulty": difficulty,
        "Blocks Mined": len(subset),
        "Average Mining Time (s)": subset["Mining Time (s)"].mean(),
        "Minimum Mining Time (s)": subset["Mining Time (s)"].min(),
        "Maximum Mining Time (s)": subset["Mining Time (s)"].max(),
        "Std Dev Mining Time (s)": subset["Mining Time (s)"].std(ddof=1),
        "Mining Throughput (hashes/s)": total_attempts / total_time,
        "Measured Avg Attempts": measured_average_attempts,
        "Theoretical Attempts (16^d)": theoretical_attempts,
        "Measured/Theoretical": measured_average_attempts / theoretical_attempts,
        "Example Valid Hash": subset.iloc[0]["Valid Hash"],
    })

# Present the summary table required by the question.
summary_df = pd.DataFrame(summary_rows)

print("\n=== REQUIRED PERFORMANCE SUMMARY ===")
print(summary_df.to_string(index=False))



=== REQUIRED PERFORMANCE SUMMARY ===
 Difficulty  Blocks Mined  Average Mining Time (s)  Minimum Mining Time (s)  Maximum Mining Time (s)  Std Dev Mining Time (s)  Mining Throughput (hashes/s)  Measured Avg Attempts  Theoretical Attempts (16^d)  Measured/Theoretical                                               Example Valid Hash
          2            30                 0.000329                 0.000005                 0.002317                 0.000458                  8.633728e+05           2.842667e+02                          256              1.110417 0017817d388aaa1bb8264f689ac0f2a61eb0e534ad92229fbe65a9d1e11f09b8
          3            30                 0.002006                 0.000028                 0.009867                 0.002539                  2.215112e+06           4.443967e+03                         4096              1.084953 000d5c893ff01c69e0034db16fae22539ec357a7525fb5fab9412b0640d777b3
          4            30                 0.024449                 0.000043  

In [10]:
# Compare measured growth with the theoretical 16x increase per zero.
growth_rows = []
for previous_difficulty, current_difficulty in zip(difficulties[:-1], difficulties[1:]):
    previous = summary_df[summary_df["Difficulty"] == previous_difficulty].iloc[0]
    current = summary_df[summary_df["Difficulty"] == current_difficulty].iloc[0]

    growth_rows.append({
        "Difficulty Step": f"{previous_difficulty} -> {current_difficulty}",
        "Average Time Ratio": (
            current["Average Mining Time (s)"] /
            previous["Average Mining Time (s)"]
        ),
        "Average Attempts Ratio": (
            current["Measured Avg Attempts"] /
            previous["Measured Avg Attempts"]
        ),
        "Theoretical Ratio": 16.0,
    })

growth_df = pd.DataFrame(growth_rows)
print("\n=== GROWTH BETWEEN DIFFICULTY LEVELS ===")
print(growth_df.to_string(index=False))

# Confirm every result satisfies its requested difficulty.
runs_df["Hash Valid"] = runs_df.apply(
    lambda row: row["Valid Hash"].startswith("0" * int(row["Difficulty"])),
    axis=1
)
print("\nAll 120 hashes satisfy their difficulty:", runs_df["Hash Valid"].all())



=== GROWTH BETWEEN DIFFICULTY LEVELS ===
Difficulty Step  Average Time Ratio  Average Attempts Ratio  Theoretical Ratio
         2 -> 3            6.093230               15.633091               16.0
         3 -> 4           12.186900               14.778629               16.0
         4 -> 5           17.075281               16.571138               16.0

All 120 hashes satisfy their difficulty: True


## Reflection Questions

### 1. Relationship between difficulty and computational effort

My results show that increasing the difficulty requires much more computational effort. The measured average number of attempts increased from **284.27** at difficulty 2 to **4,443.97** at difficulty 3, **65,675.73** at difficulty 4, and **1,088,322.00** at difficulty 5. The average mining time also increased from **0.000329 seconds** to **0.417481 seconds**.

All 120 hashes satisfied their required difficulty. The measured attempt averages do not exactly equal the theoretical values because each mining run is random, but they follow the expected relationship of approximately `16^difficulty` attempts.

### 2. Is the increase linear or exponential?

The increase is approximately exponential, not linear. The theoretical number of attempts increases by a factor of 16 for every extra leading zero. In my results, the measured attempt ratios were **15.63** from difficulty 2 to 3, **14.78** from 3 to 4, and **16.57** from 4 to 5. These values are close to the theoretical ratio of 16.

The average mining-time ratios were **6.09**, **12.19**, and **17.08**. Time does not match the attempt ratio exactly because the timing is affected by the computer and runtime, but it still increases as the difficulty rises. The standard deviation also shows variation between runs. At difficulty 5, the average time was **0.417481 seconds**, while the maximum was **1.912934 seconds**. Overall, the attempt and timing results support exponential growth rather than a linear increase.

### 3. Limitation of Proof of Work and an alternative

The main limitation I observed with Proof of Work is that it requires increasing amounts of computing power and electricity as the difficulty rises. In this experiment, the average number of attempts reached **1,088,322.00** at difficulty 5, even though only one valid hash was needed for each block. Most of those calculations were discarded.

Proof of Stake tries to address this problem by choosing validators based on the cryptocurrency they lock as a stake instead of making them compete with constant hashing. This greatly reduces energy consumption. However, Proof of Stake has its own issues, such as the possibility that users with larger stakes may have more influence over the network.
